# FinTech Agent Observability, Evaluation & Safety — Colab

Run the SecureBank multi-agent support project from Google Colab. This notebook checks out the repository, installs the core agent dependencies, configures optional API access, and lets you run the shared agent plus workshop modules A–D.

**Runtime:** Python 3.12 is recommended by the repository. Use synthetic/demo account questions only; the project includes mock customer records. API calls may incur charges. LangSmith tracing and evaluation are optional.

## 1. Get the project files

This clones the repository into the current Colab runtime. If you restart the runtime, run this cell again.

In [ ]:
from pathlib import Path
import os
from IPython.display import display, Markdown

REPO_URL = "https://github.com/baldawamanali-art/fintech-agent-observability-evaluation.git"
REPO_DIR = Path("/content/fintech-agent-observability-evaluation")

if not REPO_DIR.exists():
    !git clone --depth 1 $REPO_URL $REPO_DIR
else:
    print(f"Repository already exists at {REPO_DIR}")

%cd /content/fintech-agent-observability-evaluation
print("Python:", __import__("sys").version.split()[0])

## 2. Install the core agent dependencies

This installs the requirements needed for the shared LangGraph agent and its retrieval pipeline. Guardrails and Presidio are optional extras for Module C; their setup is shown later.

In [ ]:
%pip install -q "langchain>=0.3.14,<0.4" "langchain-community>=0.3.14" "langchain-openai>=0.3.1,<0.4" "langchain-chroma>=0.2.1,<0.3" "langsmith>=0.1.17,<0.3" "langgraph>=0.2.60,<0.3" "chromadb>=0.6.3,<0.7" "openai>=1.59.9,<2" "python-dotenv>=1.1.1" "tiktoken>=0.8.0" "setuptools<81"
print("Core dependencies installed. If imports fail, restart the Colab runtime and continue.")

## 3. Configure API keys (optional LangSmith)

Enter your OpenAI key to make the agent answer questions. The value is kept in this Colab runtime's environment and is not written into the notebook or repository. Add a LangSmith key only if you want tracing/evaluation features. You can skip LangSmith and use the agent without tracing.

In [ ]:
import os
from getpass import getpass

openai_key = getpass("OpenAI API key (required to call the agent): ").strip()
if not openai_key:
    raise ValueError("An OpenAI API key is needed to run the LLM-backed agent.")
os.environ["OPENAI_API_KEY"] = openai_key

langsmith_key = getpass("LangSmith API key (optional; press Enter to skip): ").strip()
if langsmith_key:
    os.environ["LANGCHAIN_API_KEY"] = langsmith_key
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = "fintech-support-agent-colab"
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("LangSmith tracing is off.")

## 4. Build and try the support agent

The graph classifies each question and routes it to the policy RAG agent, mock account agent, or escalation agent. The included knowledge base and account records are synthetic workshop data.

In [ ]:
import logging
import sys
from pathlib import Path

logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)
sys.path.insert(0, str(REPO_DIR / "project"))

from fintech_support_agent import build_support_agent, ask

agent = build_support_agent(
    collection_name="colab_fintech_support",
    chunk_size=1000,
    chunk_overlap=100,
    top_k=3,
)
app = agent["app"]
print("Agent is ready.")

In [ ]:
demo_queries = [
    "What is the overdraft fee?",
    "What is the balance on ACC-12345?",
    "I'm frustrated and want to speak to a manager.",
]

for query in demo_queries:
    result = ask(app, query)
    print(f"\nQuestion: {query}")
    print(f"Route: {result['intent']}")
    print(f"Answer: {result['response']}")

## 5. Run a workshop module

Choose one script path and execute it. Modules A, B, and D use the core dependencies. Module C requires additional Guardrails AI and Presidio packages. The Module B demo recreates a named LangSmith dataset, so review that script before running it if you already have LangSmith data you want to keep.

In [ ]:
# Choose one script, for example:
# "module_a_observability/demo.py"
# "module_a_observability/exercise.py"
# "module_b_evaluation/exercise.py"
# "module_c_guardrails/demo.py"  # install Module C extras first
# "module_d_cost_optimization/demo.py"
MODULE_SCRIPT = "module_a_observability/demo.py"

script_path = REPO_DIR / MODULE_SCRIPT
if not script_path.is_file():
    raise FileNotFoundError(script_path)
!python "$script_path"

### Optional Module C dependencies

The repo includes local validator wheels because the upstream Guardrails Hub installer is unavailable. The spaCy language model is large; install it only if you plan to run the Presidio exercises.

In [ ]:
# Run this cell only for Module C.
%pip install -q presidio-analyzer presidio-anonymizer spacy
!python -m pip install -q --no-deps /content/fintech-agent-observability-evaluation/vendor/guardrails_hub_wheels/*.whl
!python -m spacy download en_core_web_lg

## 6. Open the source and workshop guides

- [Repository README](https://github.com/baldawamanali-art/fintech-agent-observability-evaluation)
- [Shared support agent](https://github.com/baldawamanali-art/fintech-agent-observability-evaluation/blob/master/project/fintech_support_agent.py)
- [Module A: Observability](https://github.com/baldawamanali-art/fintech-agent-observability-evaluation/tree/master/module_a_observability)
- [Module B: Evaluation](https://github.com/baldawamanali-art/fintech-agent-observability-evaluation/tree/master/module_b_evaluation)
- [Module C: Guardrails](https://github.com/baldawamanali-art/fintech-agent-observability-evaluation/tree/master/module_c_guardrails)
- [Module D: Cost Optimization](https://github.com/baldawamanali-art/fintech-agent-observability-evaluation/tree/master/module_d_cost_optimization)

The notebook launches the existing project rather than duplicating its Python source, so repository updates remain available after rerunning the clone cell.